# Phase 1 — Notebook 00: Data Prep

**What this notebook does:** turns the raw Phase-1 legal corpus (4 huge JSON
files on Hugging Face) into two things every later Phase-1 notebook needs:

1. **`dev_prompts.json` + `dev_truth.json`** — our own held-out, test-shaped
   dataset (512-word prompt → 128-word reference continuation), built the
   *same way* the official test set is shaped. This is our "local Codabench":
   we can measure BERTScore against it as often as we want, for free, before
   spending a real submission.
2. **A clean text sample per source** (`clean_<source>.txt`) — big enough to
   train both tokenizers and run the tokenizer / data-mix ablations on, small
   enough to build in one CPU session without touching the GPU quota.

**What this notebook deliberately does *not* do:** build the final, full-size
training corpus for models A–D. That happens in a follow-up step, once the
data-mix ablation (a separate, later notebook) tells us the winning mix of
Acts / High Courts / Supreme Court / Tribunals — no point committing GPU-
scale disk and time to a mix we haven't picked yet.

**Runs on:** Kaggle CPU session (no accelerator needed — this whole notebook
never touches a GPU, so it doesn't spend any of our weekly GPU quota).


## 1. Imports

Nothing fancy here on purpose. `ijson` is the one non-standard package — it
lets us walk through a JSON array item by item without ever holding the
whole file in memory, which matters a lot once we get to the 10.7 GB
high-courts file. Everything else is standard library.


In [ ]:
# ijson streams a JSON array element-by-element instead of parsing the
# whole file into memory at once. We need this: the high-courts file alone
# is ~10.7 GB on disk, and a naive json.load() on it would try to build a
# Python object graph several times that size — more than a Kaggle CPU
# session's ~29 GB of RAM comfortably allows.
try:
    import ijson
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ijson"], check=True)
    import ijson

import hashlib      # for the "have we seen this exact document before" dedup check
import json         # writing our own small JSON files (dev set, stats, etc.)
import os
import random       # deterministic sampling — same seed, same dev set, every run
import re           # a couple of cheap regex cleanups on the raw legal text
import time         # just for progress prints, so a long cell doesn't look frozen
import unicodedata  # normalising weird unicode variants of the same character
from pathlib import Path

print("Imports OK.")


## 2. Config

Every path, budget and knob for this notebook lives here, in one place. If
we ever need to bump the clean-sample size or change where things get
written, this is the only cell that should need touching.


In [ ]:
# --- Where the raw data lives on Hugging Face -------------------------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# The four Phase-1 training sources. Keys here become the filenames we write
# later (clean_acts.txt, clean_high_courts.txt, ...), so keep them simple.
SOURCE_FILES = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# --- Where we write our own outputs -----------------------------------------
OUTPUT_DIR = Path("/kaggle/working/phase1_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Budgets ------------------------------------------------------------------
# How much *clean* text we keep per source, in characters. This is not the
# final training budget for models A-D (that comes later, after the data-mix
# ablation) — it just needs to be enough to train two tokenizers on and run
# a handful of 50M-parameter proxy runs for the ablations.
# 400M characters is roughly 70-90M words depending on the source, which is
# plenty for both jobs and still fast to stream through on CPU.
CLEAN_SAMPLE_CHARS_PER_SOURCE = 400_000_000

# A document needs at least this many words for us to be able to cut a
# 512-word prompt *and* a following 128-word reference out of it without
# running off the end. 700 gives us a small safety margin over the 640 we
# strictly need.
MIN_DOC_WORDS_FOR_DEV = 700

# Roughly 1 in every N long-enough documents gets set aside for the dev set
# instead of the clean-text sample, so the dev set is drawn from documents
# the tokenizer/ablation code never sees. How big the *final* dev set is is
# controlled separately below.
DEV_RESERVE_EVERY_N_DOCS = 500

# Target number of (prompt, continuation) pairs in the local dev set. 1,000
# is enough to make BERTScore differences between decoding strategies mean
# something, without making local evaluation slow.
DEV_SET_TARGET_SIZE = 1000

PROMPT_WORDS = 512        # matches the official test format exactly
CONTINUATION_WORDS = 128  # matches the official expected-output format

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Clean-sample budget per source: {CLEAN_SAMPLE_CHARS_PER_SOURCE:,} characters")
print(f"Dev-set target: {DEV_SET_TARGET_SIZE} pairs, {PROMPT_WORDS}-word prompt / {CONTINUATION_WORDS}-word continuation")


## 3. Hugging Face login

The dataset repo needs an HF token to download from. On Kaggle, we read it
out of a secret named `HF` (Add-ons → Secrets in the notebook editor) rather
than pasting a token into the notebook — that way the token never ends up
committed into a notebook version by accident.


In [ ]:
HF_TOKEN = None

try:
    # Only exists inside a real Kaggle kernel — guard it so this notebook
    # doesn't explode if it's ever opened somewhere else (e.g. locally).
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as exc:
    # Fall back to an environment variable, mostly so this cell doesn't
    # crash the whole notebook while poking around outside Kaggle.
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print(f"Could not read the Kaggle secret 'HF' ({exc}); "
          f"falling back to the HF_TOKEN environment variable instead.")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login OK.")
else:
    print("WARNING: no HF token found. Set the Kaggle secret 'HF' before "
          "running the download cells below, or the hf_hub_download calls "
          "will fail on the private/rate-limited paths.")


## 4. Download the raw files

`hf_hub_download` is smart about this: if a file has already been downloaded
into the local HF cache (e.g. we re-run this cell after a crash), it won't
re-download it. That matters a lot for the 10.7 GB high-courts file.


In [ ]:
from huggingface_hub import hf_hub_download

# We deliberately do this in a loop with a dict, not four copy-pasted cells —
# one function, four inputs, one thing to fix if the download logic ever
# needs to change.
local_paths = {}

for name, repo_path in SOURCE_FILES.items():
    print(f"Fetching {name} ({repo_path}) ...")
    t0 = time.time()
    local_paths[name] = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    print(f"  -> {local_paths[name]}  ({time.time() - t0:.1f}s)")

test_local_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"Test file -> {test_local_path}")


## 5. Peek at the schema before doing anything else

Before writing any real processing code, let's actually look at a couple of
real records from each file. This is the "Step 1 - read the data format"
step the capstone brief itself recommends, and it's cheap even on the huge
files because `ijson` never has to read past the first couple of records.


In [ ]:
def peek_first_records(path, n=1):
    """Yield the first `n` records of a JSON file shaped like a list of
    objects, e.g. [{"text": "..."}, {"text": "..."}, ...], without ever
    reading the rest of the file. Safe to call on the 10 GB files.
    """
    with open(path, "rb") as f:
        for i, record in enumerate(ijson.items(f, "item")):
            if i >= n:
                break
            yield record


for name, path in local_paths.items():
    print(f"--- {name} ---")
    for record in peek_first_records(path, n=1):
        keys = list(record.keys())
        text_preview = record.get("text", "")[:300].replace("\n", " ")
        print(f"  keys: {keys}")
        print(f"  text[:300]: {text_preview!r}")
    print()

# The test file is small (a few MB), so json.load is fine here — no need
# for the streaming machinery.
with open(test_local_path, encoding="utf-8") as f:
    test_data = json.load(f)

first_test_id = next(iter(test_data))
first_test_words = len(test_data[first_test_id]["text"].split())
print(f"Test file: {len(test_data)} prompts, first prompt id={first_test_id!r}, "
      f"{first_test_words} words (expect ~512).")


## 6. Cleaning utilities

Three small, single-purpose functions instead of one big "clean everything"
function — each one is easy to test on its own, and easy to swap out later
if we decide a rule is too aggressive (or not aggressive enough).


In [ ]:
_WHITESPACE_RE = re.compile(r"[ \t ]+")   # runs of spaces/tabs/nbsp
_BLANK_LINES_RE = re.compile(r"\n{3,}")         # 3+ blank lines in a row


def normalize_text(text):
    """Unicode-normalise and tidy up whitespace, without touching the
    actual words. This alone fixes a surprising number of "duplicate"
    documents that differ only by curly quotes, non-breaking spaces, or a
    stray extra blank line between paragraphs.
    """
    text = unicodedata.normalize("NFKC", text)
    text = _WHITESPACE_RE.sub(" ", text)
    text = _BLANK_LINES_RE.sub("\n\n", text)
    return text.strip()


def looks_like_junk(text, min_words=50, min_alpha_ratio=0.5):
    """Cheap heuristic filter for documents that aren't really legal text:
    near-empty scraps, or pages that are mostly page numbers / punctuation /
    OCR noise rather than prose. Two simple checks, not a full classifier —
    good enough to drop the obvious junk without accidentally eating real
    (if unusually formatted) judgments.
    """
    words = text.split()
    if len(words) < min_words:
        return True
    alpha_chars = sum(1 for ch in text if ch.isalpha())
    if alpha_chars / max(len(text), 1) < min_alpha_ratio:
        return True
    return False


def doc_fingerprint(text):
    """A short, fixed-size fingerprint of a document's normalised text,
    used purely for exact-duplicate detection. We use blake2b at 16 bytes
    (not the full text) so the "seen it before" set stays small even across
    several million documents — a set of a few million 16-byte digests is a
    fraction of a GB, versus keeping full documents around to compare.
    """
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).digest()


print("Cleaning utilities ready.")


## 7. The streaming reader

This is the one function every source file goes through. It:

1. Walks the file with `ijson` (constant memory, regardless of file size).
2. Normalises + filters junk out of each document.
3. Skips exact duplicates it's already seen (within *this* source — cross-
   source dedup isn't worth the extra bookkeeping at this budget).
4. Yields `(text, word_count)` pairs, one document at a time.

Nothing downstream ever calls `json.load()` on the big files — this
generator is the only thing that touches them.


In [ ]:
def stream_clean_documents(path, seen_hashes):
    """Generator: yields (clean_text, word_count) for every document in
    `path` that survives cleaning + dedup. `seen_hashes` is a set the caller
    owns and passes in, so dedup state can be shared or reset per-source as
    needed.
    """
    with open(path, "rb") as f:
        for record in ijson.items(f, "item"):
            raw_text = record.get("text")
            if not raw_text:
                continue

            text = normalize_text(raw_text)
            if looks_like_junk(text):
                continue

            fp = doc_fingerprint(text)
            if fp in seen_hashes:
                continue
            seen_hashes.add(fp)

            yield text, len(text.split())


print("stream_clean_documents() ready.")


## 8. Build the dev set *and* the clean text sample in one pass

Both outputs come from the same stream, so we only read each huge file
once. For every long-enough document, we flip a coin (roughly 1-in-`
DEV_RESERVE_EVERY_N_DOCS`): heads, it becomes a dev-set (prompt,
continuation) pair; tails, its full text goes into the clean-sample file
for that source. A document only ever contributes to one or the other,
which keeps the dev set honestly held out from the tokenizer/ablation data.

We stop reading a source as soon as we've hit *both* budgets for it, so a
source we don't need much of (e.g. tribunals, which the plan doc flags as
noisier) doesn't cost us more streaming time than it has to.


In [ ]:
def cut_prompt_and_continuation(words, prompt_words=PROMPT_WORDS, cont_words=CONTINUATION_WORDS):
    """Slice a document's word list into a prompt and the reference
    continuation that immediately follows it — exactly how the official
    test set is shaped (512 words in, 128-word continuation as the answer).
    """
    prompt = " ".join(words[:prompt_words])
    continuation = " ".join(words[prompt_words:prompt_words + cont_words])
    return prompt, continuation


dev_prompts = {}   # id -> {"text": "<512 word prompt>"}          (mirrors the official test format)
dev_truth = {}      # id -> {"text": "<128 word continuation>"}    (mirrors evaluate_phase1.py's truth format)
dev_counter = 0

per_source_stats = {}

for source_name, path in local_paths.items():
    print(f"--- Streaming {source_name} ---")
    t0 = time.time()

    seen_hashes = set()
    clean_chars_written = 0
    docs_seen = 0
    docs_kept_for_sample = 0
    dev_pairs_from_this_source = 0

    out_path = OUTPUT_DIR / f"clean_{source_name}.txt"
    with open(out_path, "w", encoding="utf-8") as out_f:
        for text, n_words in stream_clean_documents(path, seen_hashes):
            docs_seen += 1

            # Every DEV_RESERVE_EVERY_N_DOCS-th long-enough document is
            # reserved for the dev set instead of the clean sample, so the
            # two outputs never overlap.
            is_dev_candidate = (
                n_words >= MIN_DOC_WORDS_FOR_DEV
                and docs_seen % DEV_RESERVE_EVERY_N_DOCS == 0
                and len(dev_prompts) < DEV_SET_TARGET_SIZE
            )

            if is_dev_candidate:
                words = text.split()
                prompt, continuation = cut_prompt_and_continuation(words)
                dev_id = f"{source_name}_{dev_counter:05d}"
                dev_prompts[dev_id] = {"text": prompt}
                dev_truth[dev_id] = {"text": continuation}
                dev_counter += 1
                dev_pairs_from_this_source += 1
                continue  # this document is spent; it does not also go into the clean sample

            if clean_chars_written < CLEAN_SAMPLE_CHARS_PER_SOURCE:
                out_f.write(text)
                out_f.write("\n\n")
                clean_chars_written += len(text) + 2
                docs_kept_for_sample += 1

            # Stop early once both budgets for this source are satisfied —
            # no reason to keep streaming a 10 GB file after we already
            # have what we need from it.
            sample_full = clean_chars_written >= CLEAN_SAMPLE_CHARS_PER_SOURCE
            dev_full_overall = len(dev_prompts) >= DEV_SET_TARGET_SIZE
            if sample_full and dev_full_overall:
                break

    elapsed = time.time() - t0
    per_source_stats[source_name] = {
        "docs_seen": docs_seen,
        "docs_kept_for_sample": docs_kept_for_sample,
        "clean_chars_written": clean_chars_written,
        "dev_pairs_contributed": dev_pairs_from_this_source,
        "seconds": round(elapsed, 1),
    }
    print(f"  docs seen: {docs_seen:,} | kept for sample: {docs_kept_for_sample:,} "
          f"| clean chars: {clean_chars_written:,} | dev pairs: {dev_pairs_from_this_source} "
          f"| {elapsed:.1f}s")

print()
print(f"Total dev-set pairs collected: {len(dev_prompts)} (target was {DEV_SET_TARGET_SIZE})")


## 9. Write the dev set to disk

Written in the *exact* shape `evaluate_phase1.py` expects — `dev_prompts.json`
matches what a real submission's `prediction.json` looks like, and
`dev_truth.json` matches what that script wants as `--truth`. That means we
can run the official-style local evaluator against our own predictions with
zero extra glue code, any time we want a BERTScore number before submitting.


In [ ]:
dev_prompts_path = OUTPUT_DIR / "dev_prompts.json"
dev_truth_path = OUTPUT_DIR / "dev_truth.json"

with open(dev_prompts_path, "w", encoding="utf-8") as f:
    json.dump(dev_prompts, f, ensure_ascii=False, indent=2)

with open(dev_truth_path, "w", encoding="utf-8") as f:
    json.dump(dev_truth, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(dev_prompts)} pairs to:")
print(f"  {dev_prompts_path}")
print(f"  {dev_truth_path}")
print()
print("Sanity check — same ID set on both sides (evaluate_phase1.py requires this):")
print(set(dev_prompts.keys()) == set(dev_truth.keys()))


## 10. Sanity-check everything before we call this done

A few quick checks that would each have caught a real bug while this
notebook was being written — cheap insurance against silently shipping a
broken dev set or an empty clean-text file downstream.


In [ ]:
import textwrap

print("=== Per-source stats ===")
for name, stats in per_source_stats.items():
    print(f"{name:14s} {stats}")

print()
print("=== Clean-sample file sizes on disk ===")
for name in SOURCE_FILES:
    p = OUTPUT_DIR / f"clean_{name}.txt"
    size_mb = p.stat().st_size / 1e6 if p.exists() else 0.0
    print(f"  clean_{name}.txt: {size_mb:,.1f} MB")

print()
print("=== A random dev pair, just to eyeball it ===")
sample_id = random.choice(list(dev_prompts.keys()))
print(f"id: {sample_id}")
print("prompt (last 200 chars):")
print(textwrap.fill(dev_prompts[sample_id]["text"][-200:], width=100))
print("continuation:")
print(textwrap.fill(dev_truth[sample_id]["text"], width=100))

# The prompt should end right where the continuation begins, word-for-word —
# if this assert ever fails, the prompt/continuation cut logic has a bug.
combined_check = (dev_prompts[sample_id]["text"] + " " + dev_truth[sample_id]["text"]).split()
assert len(combined_check[:PROMPT_WORDS]) == PROMPT_WORDS, "prompt is the wrong length"
print()
print("All checks passed.")


## 11. Package for reuse in later notebooks

We don't push to a Kaggle Dataset from inside the notebook (that needs the
`kaggle` CLI + an API token, which is its own bit of setup) — instead, this
cell just lists exactly what to attach when creating one, either through the
"New Dataset" button on `/kaggle/working/phase1_data`, or with the Kaggle
CLI from outside the notebook.

**What to publish as the `legallm-phase1-data` Kaggle Dataset:**
- `dev_prompts.json`, `dev_truth.json` — our local test-like dev set
- `clean_<source>.txt` × 4 — used next by the tokenizer-training and
  data-mix-ablation notebooks

**What happens next:** the tokenizer-training notebook reads the four
`clean_*.txt` files to train T1 (BPE) and T2 (Unigram); the data-mix
ablation notebook samples from them in different ratios to train the three
50M-parameter proxy models; and once a winning mix is picked, a follow-up
"build final corpus" step reuses `stream_clean_documents()` from this
notebook (copy the function, or better, move it into the shared `legallm`
package) to build the full-size token shards for models A-D.


In [ ]:
print("Files ready in", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:28s} {size_mb:10,.1f} MB")

print()
print("Next: create a Kaggle Dataset from this folder (or the CLI equivalent),")
print("then attach it as an input to the tokenizer-training notebook.")
